**Import Libraries**

In [33]:
import pandas as pd
import numpy as np
import pandas as pd
import pickle
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

**Load the Dataset**

In [6]:
df=pd.read_parquet("green_tripdata_2026-01.parquet")
df.describe()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
count,40272.000000,40272,40272,34858.000000,40272.000000,40272.000000,34858.000000,40272.000000,40272.000000,40272.000000,40272.000000,40272.000000,40272.000000,0.0,40272.000000,40272.000000,34858.000000,34857.000000,34858.000000,40272.000000
mean,2.279897,2026-01-16 22:38:57.806366,2026-01-16 22:59:02.332389,1.207212,96.190480,141.878352,1.298382,12.551243,16.006925,0.823550,0.561687,2.476605,0.260190,NaN,0.922189,24.195461,1.259711,1.046562,0.851383,0.058179
min,1.000000,2025-12-27 16:49:41,2025-12-27 17:02:11,1.000000,1.000000,1.000000,0.000000,0.000000,-70.000000,-7.500000,-0.500000,-2.520000,0.000000,NaN,-1.000000,-76.500000,1.000000,1.000000,0.000000,0.000000
25%,2.000000,2026-01-09 11:18:24,2026-01-09 11:37:25.250000,1.000000,74.000000,74.000000,1.000000,1.200000,8.600000,0.000000,0.500000,0.000000,0.000000,NaN,1.000000,14.600000,1.000000,1.000000,0.000000,0.000000
50%,2.000000,2026-01-16 14:27:16.500000,2026-01-16 14:47:26.500000,1.000000,75.000000,140.000000,1.000000,1.960000,12.800000,0.000000,0.500000,2.000000,0.000000,NaN,1.000000,20.000000,1.000000,1.000000,0.000000,0.000000
75%,2.000000,2026-01-23 19:37:11.500000,2026-01-23 19:55:24.250000,1.000000,97.000000,229.000000,1.000000,3.470000,19.100000,1.000000,0.500000,3.770000,0.000000,NaN,1.000000,28.860000,1.000000,1.000000,2.750000,0.000000
max,6.000000,2026-02-01 21:08:36,2026-02-01 21:15:02,99.000000,265.000000,265.000000,9.000000,179830.920000,960.000000,7.500000,4.250000,300.000000,85.000000,NaN,1.000000,961.000000,4.000000,2.000000,2.750000,0.750000
std,1.233095,NaN,NaN,1.018101,55.683047,77.583150,0.950439,1033.875580,14.951959,1.332193,0.321908,3.564317,1.583427,NaN,0.245158,17.177547,0.471217,0.210701,1.271401,0.200626


**Feature Engineering**

In [14]:
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

In [17]:
features = ['PU_DO', 'trip_distance']

train_dicts = df[features].to_dict(orient='records')


**Vectorize the Features**

In [18]:
dv = DictVectorizer()

X = dv.fit_transform(train_dicts)

In [20]:
y = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60

**Train the Baseline Model**

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

**Evaluate the Model**

In [28]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

MAE  : 13.5417
RMSE : 76.2646


**Save the Fitted Model**

In [34]:
with open("models/baseline.pkl", "wb") as f:
    pickle.dump(model, f)